# Download the NYC 311 models for review
Run the next cell in Colab, then attach the downloaded ZIP in chat. It verifies each model against its saved SHA-256 and includes the records needed to check versions and lineage. It does not include raw request data.


In [ ]:
# Verify and download only the model files and their records (no raw data).
from google.colab import drive, files
from pathlib import Path
import hashlib, json, zipfile

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
run_id = "20260926T120702352046Z"
experiment_id = "20260926T143147282446Z"
final_id = "20260926T143549799163Z"
decision_id = "20260926T143947829974Z"

run_dir = root / "runs" / run_id
experiment_dir = root / "experiments" / experiment_id
final_dir = root / "final_evaluations" / final_id
decision_dir = root / "release_decisions" / decision_id
run_file = run_dir / "run.json"
experiment_file = experiment_dir / "experiment.json"
plan_file = final_dir / "locked_plan.json"
report_file = final_dir / "test_report.json"
decision_file = decision_dir / "decision.json"

run = json.loads(run_file.read_text())
experiment = json.loads(experiment_file.read_text())
plan = json.loads(plan_file.read_text())
report = json.loads(report_file.read_text())
decision = json.loads(decision_file.read_text())

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

if run["run_id"] != run_id or experiment["experiment_id"] != experiment_id:
    raise ValueError("Wrong run or experiment")
if plan["final_id"] != final_id or report["final_id"] != final_id:
    raise ValueError("Wrong final evaluation")
if decision["decision_id"] != decision_id or decision["decision"] != "HOLD_FOR_OPERATIONAL_USE":
    raise ValueError("Wrong release decision")
for path, key in [(plan_file, "locked_plan_sha256"),
                  (report_file, "test_report_sha256"),
                  (experiment_file, "experiment_sha256")]:
    if sha256(path) != decision[key]:
        raise ValueError(f"Changed record: {path.name}")

models = {
    "initial_logistic/model.joblib": (run_dir / "model.joblib", run["model_sha256"]),
    "final_tree/model.joblib": (final_dir / "model.joblib", plan["model_sha256"]),
}
for name in ["logistic", "tree"]:
    info = experiment["artifacts"][name]
    models[f"candidates/{name}.joblib"] = (experiment_dir / info["file"], info["sha256"])

for name, (path, expected) in models.items():
    if sha256(path) != expected:
        raise ValueError(f"Changed model: {name}")
    print(name, path.stat().st_size, "bytes", expected)

archive_path = Path("/content/nyc311_models_for_review.zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name, (path, _) in models.items():
        archive.write(path, name)
    for name, path in {
        "initial_logistic/run.json": run_file,
        "candidates/experiment.json": experiment_file,
        "final_tree/locked_plan.json": plan_file,
        "final_tree/test_report.json": report_file,
        "decision/decision.json": decision_file,
    }.items():
        archive.write(path, name)

print("ZIP:", archive_path)
print("ZIP SHA-256:", sha256(archive_path))
print("No raw request data included")
files.download(str(archive_path))
